In [3]:
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

In [4]:
# --- CONFIGURACIÓN DE INFLUXDB (v2) ---
env_file = find_dotenv(".env", usecwd=True)
if not env_file:
   for candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"):
       if candidate.is_file():
           env_file = str(candidate)
           break


if not env_file:
   raise FileNotFoundError("No se encontró el archivo .env")


load_dotenv(env_file, override=True)


def required_env(name):
   value = os.getenv(name)
   if not value:
       raise RuntimeError(f"Falta la variable {name} en {env_file}")
   return value


INFLUX_URL = required_env("INFLUX_URL")
INFLUX_TOKEN = required_env("INFLUX_TOKEN")
INFLUX_ORG = required_env("INFLUX_ORG")
INFLUX_BUCKET_DOBOT = required_env("INFLUX_BUCKET_DOBOT")
# Busca automáticamente el archivo .env en el directorio actual

# --- PARÁMETROS DEL DATASET ---
CSV_FILE = "../datos/dataset_dobot_cr5a.csv"
DEVICE_NAME = "Dobot_CR5A"                 # Se asignará como Tag

In [5]:
print(f"Cargando dataset desde '{CSV_FILE}'...")

try:
    df = pd.read_csv(CSV_FILE)
    
    # Parsear la estampa de tiempo a objetos datetime nativos
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    print(f"Registros cargados: {len(df)} filas.")
    print("\nVista previa de los datos:")
    display(df.head(3))

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{CSV_FILE}'. Revisa la ruta.")

Cargando dataset desde '../datos/dataset_dobot_cr5a.csv'...
Registros cargados: 987 filas.

Vista previa de los datos:


,timestamp,robot_mode,pos_x,pos_y,pos_z,rot_rx,rot_ry,rot_rz,j1_temp,j2_temp,j3_temp,j4_temp,j5_temp,j6_temp,j1_current,j2_current,j3_current,j4_current,j5_current,j6_current
0,2026-10-07 09:21:21.879468,5,25.47,68.25,-2.5,-87.75,30.52,-0.0,-663.2,223.6,177.7,1.6,143.4,-0.0,-0.144,-0.512,0.041,-0.004,0.000,0.0
1,2026-10-07 09:21:22.375259,5,25.47,68.25,-2.5,-87.75,30.52,-0.0,-663.2,223.6,177.7,1.6,143.4,0.0,0.217,0.393,-0.031,0.001,0.068,0.0
2,2026-10-07 09:21:22.876674,5,25.47,68.25,-2.5,-87.75,30.52,-0.0,-663.2,223.6,177.7,1.6,143.4,-0.0,-0.042,0.007,-0.012,-0.000,0.068,0.0


In [6]:
# Inicializar cliente de InfluxDB y la API de escritura síncrona
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

puntos_batch = []
tamanio_lote = 500  # Envío agrupado de 500 filas por llamada para maximizar velocidad
total_filas = len(df)

print(f"Subiendo {total_filas} registros al bucket '{INFLUX_BUCKET_DOBOT}'...")

for index, row in df.iterrows():
    punto = (
        Point("dobot_telemetry")  # Measurement
        
        # --- TAGS (Metadatos de filtrado rápido) ---
        .tag("device_name", DEVICE_NAME)
        .tag("model", "CR5A")
        
        # --- FIELDS: Posición Cartesiana de la Herramienta (mm / deg) ---
        .field("pos_x", float(row["pos_x"]))
        .field("pos_y", float(row["pos_y"]))
        .field("pos_z", float(row["pos_z"]))
        .field("rot_rx", float(row["rot_rx"]))
        .field("rot_ry", float(row["rot_ry"]))
        .field("rot_rz", float(row["rot_rz"]))
        
        # --- FIELDS: Temperaturas de los 6 Servomotores (°C) ---
        .field("j1_temp", float(row["j1_temp"]))
        .field("j2_temp", float(row["j2_temp"]))
        .field("j3_temp", float(row["j3_temp"]))
        .field("j4_temp", float(row["j4_temp"]))
        .field("j5_temp", float(row["j5_temp"]))
        .field("j6_temp", float(row["j6_temp"]))
        
        # --- FIELDS: Corrientes de Consumo de los 6 Ejes (A) ---
        .field("j1_current", float(row["j1_current"]))
        .field("j2_current", float(row["j2_current"]))
        .field("j3_current", float(row["j3_current"]))
        .field("j4_current", float(row["j4_current"]))
        .field("j5_current", float(row["j5_current"]))
        .field("j6_current", float(row["j6_current"]))
        
        # --- FIELDS: Estado Operativo del Robot ---
        .field("robot_mode", int(row["robot_mode"]))
        
        # --- TIMESTAMP ---
        .time(row["timestamp"], WritePrecision.NS)
    )
    puntos_batch.append(punto)

    # Enviar al servidor al completar un lote
    if len(puntos_batch) >= tamanio_lote:
        write_api.write(bucket=INFLUX_BUCKET_DOBOT, org=INFLUX_ORG, record=puntos_batch)
        puntos_batch = []
        print(f"Registros cargados: {index + 1}/{total_filas}...")

# Enviar los datos sobrantes del último lote
if puntos_batch:
    write_api.write(bucket=INFLUX_BUCKET_DOBOT, org=INFLUX_ORG, record=puntos_batch)

print(f"\n¡Migración completada! {total_filas} filas insertadas en InfluxDB.")
client.close()

Subiendo 987 registros al bucket 'dobot'...
Registros cargados: 500/987...

¡Migración completada! 987 filas insertadas en InfluxDB.
